# Calories Burnt Prediction

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Load Dataset

In [ ]:
calories = pd.read_csv('calories.csv')
exercise = pd.read_csv('exercise.csv')

print('Calories dataset shape:', calories.shape)
print('Exercise dataset shape:', exercise.shape)

In [ ]:
calories.head()

In [ ]:
exercise.head()

In [ ]:
# Merging both datasets on User_ID
df = pd.merge(exercise, calories, on='User_ID')
print('Merged dataset shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Data Cleaning

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Check and remove duplicate rows
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

In [ ]:
# Dropping User_ID as it's just an identifier and not useful for prediction
df.drop(columns=['User_ID'], inplace=True)
df.head()

In [ ]:
# Encoding Gender column (male=1, female=0)
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df.head()

## EDA (Exploratory Data Analysis)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Distribution of Calories burnt
plt.figure(figsize=(7, 4))
plt.hist(df['Calories'], bins=30, color='steelblue', edgecolor='black')
plt.title('Distribution of Calories Burnt')
plt.xlabel('Calories')
plt.ylabel('Count')
plt.show()

In [ ]:
# Duration vs Calories
plt.figure(figsize=(7, 4))
plt.scatter(df['Duration'], df['Calories'], alpha=0.4, color='coral')
plt.title('Exercise Duration vs Calories Burnt')
plt.xlabel('Duration (minutes)')
plt.ylabel('Calories')
plt.show()

In [ ]:
# Heart Rate vs Calories
plt.figure(figsize=(7, 4))
plt.scatter(df['Heart_Rate'], df['Calories'], alpha=0.4, color='green')
plt.title('Heart Rate vs Calories Burnt')
plt.xlabel('Heart Rate')
plt.ylabel('Calories')
plt.show()

In [ ]:
# Average calories burnt by gender
plt.figure(figsize=(5, 4))
df.groupby('Gender')['Calories'].mean().plot(kind='bar', color=['coral', 'steelblue'], edgecolor='black')
plt.title('Average Calories Burnt by Gender')
plt.xlabel('Gender (0=Female, 1=Male)')
plt.ylabel('Avg Calories')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Age vs Calories
plt.figure(figsize=(7, 4))
plt.scatter(df['Age'], df['Calories'], alpha=0.4, color='purple')
plt.title('Age vs Calories Burnt')
plt.xlabel('Age')
plt.ylabel('Calories')
plt.show()

## Model Training

In [ ]:
# Splitting features and target
X = df.drop(columns=['Calories'])
y = df['Calories']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('Linear Regression')
print('R2 Score:', round(r2_score(y_test, y_pred_lr), 4))
print('MAE:', round(mean_absolute_error(y_test, y_pred_lr), 4))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred_lr)), 4))

In [ ]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('Decision Tree')
print('R2 Score:', round(r2_score(y_test, y_pred_dt), 4))
print('MAE:', round(mean_absolute_error(y_test, y_pred_dt), 4))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred_dt)), 4))

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Random Forest')
print('R2 Score:', round(r2_score(y_test, y_pred_rf), 4))
print('MAE:', round(mean_absolute_error(y_test, y_pred_rf), 4))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred_rf)), 4))

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print('Gradient Boosting')
print('R2 Score:', round(r2_score(y_test, y_pred_gb), 4))
print('MAE:', round(mean_absolute_error(y_test, y_pred_gb), 4))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred_gb)), 4))

## Model Comparison

In [ ]:
models = ['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting']
r2_scores = [
    r2_score(y_test, y_pred_lr),
    r2_score(y_test, y_pred_dt),
    r2_score(y_test, y_pred_rf),
    r2_score(y_test, y_pred_gb)
]

plt.figure(figsize=(8, 4))
plt.bar(models, r2_scores, color=['steelblue', 'coral', 'green', 'purple'])
plt.title('Model Comparison - R2 Score')
plt.ylabel('R2 Score')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Actual vs Predicted - Gradient Boosting (best model)
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred_gb, alpha=0.5, color='steelblue')
plt.plot([0, 320], [0, 320], 'r--')
plt.title('Actual vs Predicted - Gradient Boosting')
plt.xlabel('Actual Calories')
plt.ylabel('Predicted Calories')
plt.show()

In [ ]:
# Feature Importance from Random Forest
importances = rf.feature_importances_
feature_names = X.columns

plt.figure(figsize=(7, 4))
plt.barh(feature_names, importances, color='steelblue')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance Score')
plt.show()

## Predict on New Data

In [ ]:
# Gender encoding: male=1, female=0
# Input: Gender, Age, Height, Weight, Duration, Heart_Rate, Body_Temp

new_person = [[1, 25, 175.0, 75.0, 20.0, 100.0, 40.2]]  # Male, 25 yrs, 175cm, 75kg, 20 min, HR=100, Temp=40.2

predicted_calories = gb.predict(new_person)
print('Predicted Calories Burnt:', round(predicted_calories[0], 2))

## Summary

In this project we predicted the number of calories burnt during exercise using machine learning.

We started by loading two separate datasets — exercise and calories — and merged them on the User_ID column. The combined dataset had 15000 rows and 9 columns. After merging, we cleaned the data by checking for missing values and duplicates. Since there were none, the dataset was already in good shape. We dropped the User_ID column as it has no predictive value and encoded the Gender column using LabelEncoder.

For EDA, we plotted a correlation heatmap and several scatter plots. We found that Duration, Heart_Rate and Body_Temp are the most strongly correlated features with Calories burnt, which makes intuitive sense.

We trained four models — Linear Regression, Decision Tree, Random Forest and Gradient Boosting. After comparing their R2 scores, Gradient Boosting gave the best results overall. Finally we used it to predict calories burnt for a new input sample.